<a href="https://colab.research.google.com/github/joseguilhermemarinho/big_data_proj/blob/main/codigo/ingestao_av1_big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import requests
import pandas as pd
from datetime import datetime
import pytz
import os

url = "https://dados.mobilidade.rio/gps/sppo"
fuso_horario = pytz.timezone('America/Recife')

In [24]:
response = requests.get(url)

if response.status_code == 200:
  dados_brutos = response.json()
  df_gps = pd.DataFrame(dados_brutos)

  df_gps['momento_extracao'] = datetime.now(fuso_horario).strftime('%Y-%m-%d %H:%M:%S')

  nome_arquivo = "gps_sppo_bruto.csv"
  df_gps.to_csv(nome_arquivo, index=False)

  print(f"Foram ingeridos {len(df_gps)} registros de GPS.")
  display(df_gps.head())

else:
  print(f"Erro ao acessar a API. Status: {response.status_code}")

Foram ingeridos 489006 registros de GPS.


,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao
0,D33258,"-22,91118","-43,19123",1775586844000,19,383,1775586847000,1775586848000,2026-04-07 16:34:17
1,D33175,"-22,87352","-43,46544",1775586843000,16,812,1775586847000,1775586848000,2026-04-07 16:34:17
2,D33149,"-22,89648","-43,61057",1775586843000,31,757,1775586847000,1775586848000,2026-04-07 16:34:17
3,D33164,"-22,8307","-43,34547",1775586843000,17,753,1775586847000,1775586848000,2026-04-07 16:34:17
4,B58057,"-22,90564","-43,20857",1775586838000,36,312,1775586847000,1775586851000,2026-04-07 16:34:17


aacessa os dados via API, acrescenta o momento do salvamento e cria o primeiro arquivo bruto. porém, o arquivo bruto é muito grande, precisamos garantir que não vai dar trabalho à equipe de transformação.

In [25]:
tamanho_mb = os.path.getsize("gps_sppo_bruto.csv") / (1024 * 1024)
print(f"O arquivo original tem: {tamanho_mb:.2f} MB")

nome_amostra_github = "amostra_gps_sppo_bruto.csv"
df_gps.head(2000).to_csv(nome_amostra_github, index=False)

print(f"Amostra reduzida com sucesso")

O arquivo original tem: 46.52 MB
Amostra reduzida com sucesso


a extração completa resultou em quase meio milhão de registros, o que gera um arquivo muito pesado. a redução para 2000 linhas foi aplicada para cumprir estritamente as diretrizes do projeto, que determinam que a pasta /dados deve conter apenas amostras pequenas e que arquivos grandes não devem ser commitados. isso garante a performance do repositório no GitHub e evita bloqueios de tamanho.

In [26]:
df = df_gps.copy()

colunas_timestamp = ['datahora', 'datahoraenvio', 'datahoraservidor']

for col in colunas_timestamp:
  df[col] = pd.to_datetime(df[col], unit='ms', utc=True)
  df[col] = df[col].dt.tz_convert('America/Sao_Paulo')

print("Exemplo após conversão:")
print(df[['datahora', 'datahoraenvio', 'datahoraservidor']].head())

/tmp/ipykernel_24161/3808728741.py:6: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df[col] = pd.to_datetime(df[col], unit='ms', utc=True)
/tmp/ipykernel_24161/3808728741.py:6: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df[col] = pd.to_datetime(df[col], unit='ms', utc=True)


Exemplo após conversão:
                   datahora             datahoraenvio  \
0 2026-04-07 15:34:04-03:00 2026-04-07 15:34:07-03:00   
1 2026-04-07 15:34:03-03:00 2026-04-07 15:34:07-03:00   
2 2026-04-07 15:34:03-03:00 2026-04-07 15:34:07-03:00   
3 2026-04-07 15:34:03-03:00 2026-04-07 15:34:07-03:00   
4 2026-04-07 15:33:58-03:00 2026-04-07 15:34:07-03:00   

           datahoraservidor  
0 2026-04-07 15:34:08-03:00  
1 2026-04-07 15:34:08-03:00  
2 2026-04-07 15:34:08-03:00  
3 2026-04-07 15:34:08-03:00  
4 2026-04-07 15:34:11-03:00  


/tmp/ipykernel_24161/3808728741.py:6: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df[col] = pd.to_datetime(df[col], unit='ms', utc=True)


Os dados nos são enviados em tempo real. A cada vez que executamos a célula que puxa os dados da api, são enviados novos dados para nós.

As colunas de datahora, datahoraenvio e datahoraservidor estão todas em um timestamp unix normalmente utilizados em sistema iot, o que acontece com os dados desse datalake.

Por isso, a conversão desse timestamp para o formato de yyyy-mm-dd hh:mm:ss - GMT-3 é muito importante para que nós possamos visualizar melhor e também entendermos perfeitamente a "tradução" do timestamp.

In [27]:
#Criando coluinas derivadas a partir do datahora

df['data'] = df['datahora'].dt.date
df['hora'] = df['datahora'].dt.hour
df['minuto'] = df['datahora'].dt.minute
df['dia_semana'] = df['datahora'].dt.day_name()

#Calculando latencia de envio: tempo entre captura do GPS e envio (ms)
df['latencia_envio_s'] = (
    df['datahoraenvio'] - df['datahora']
).dt.total_seconds()

#Calculando agora latencia do servidor: tempo entre envio e chegada ao servidor (ms)
df['latencia_servidor_s'] = (
    df['datahoraservidor'] - df['datahoraenvio']
).dt.total_seconds()

print("\nLatências calculadas:")
print(df[['latencia_envio_s', 'latencia_servidor_s']].describe())


Latências calculadas:
       latencia_envio_s  latencia_servidor_s
count     489006.000000        489006.000000
mean          16.504235            16.329401
std          117.971690             9.147626
min            0.000000          -703.000000
25%            5.000000             9.000000
50%            8.000000            16.000000
75%           10.000000            24.000000
max        12783.000000            46.000000


In [28]:
print("\nNulos antes da limpeza:")
print(df.isnull().sum())


Nulos antes da limpeza:
ordem                  0
latitude               0
longitude              0
datahora               0
velocidade             0
linha                  0
datahoraenvio          0
datahoraservidor       0
momento_extracao       0
data                   0
hora                   0
minuto                 0
dia_semana             0
latencia_envio_s       0
latencia_servidor_s    0
dtype: int64


Por mais que não existam números nulos antes da limpeza, é importante lembrar que os dados que estamos tratando são todos em tempo real. A cada nova execução de linha de código, novos dados chegam.

Por isso, precisamos normalizar, padronizar e estruturar corretamente cada dado novo que chega pois isso facilitará e muito na nossa análise da camada gold.

In [29]:
# Substituir vírgula por ponto ANTES de converter para numérico
for col in ['latitude', 'longitude', 'velocidade']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop de linhas com nulos em colunas críticas
colunas_criticas = ['ordem', 'latitude', 'longitude', 'datahora', 'velocidade']
df = df.dropna(subset=colunas_criticas)

# Remover coordenadas fora do Brasil
df = df[
    (df['latitude'].between(-34, 5)) &
    (df['longitude'].between(-74, -28))
]

# Remover velocidades negativas
df = df[df['velocidade'] >= 0]

print("Shape após limpeza:", df.shape)
print("\nExemplo de coordenadas após correção:")
print(df[['ordem', 'latitude', 'longitude', 'velocidade']].head())

Shape após limpeza: (489006, 15)

Exemplo de coordenadas após correção:
    ordem  latitude  longitude  velocidade
0  D33258 -22.91118  -43.19123          19
1  D33175 -22.87352  -43.46544          16
2  D33149 -22.89648  -43.61057          31
3  D33164 -22.83070  -43.34547          17
4  B58057 -22.90564  -43.20857          36


In [32]:
#Salvando na camada silver

os.makedirs('/content/silver', exist_ok=True)

df.to_parquet('/content/silver/sppo_silver.parquet', index=False)
df.to_csv('/content/silver/sppo_silver.csv', index=False)

df_amostra = df.sample(n=1000, random_state=42)
df_amostra.to_csv('/content/silver/sppo_amostra.csv', index=False)

print("Salvo em /content/silver/")
print("Shape final:", df.shape)

Salvo em /content/silver/
Shape final: (489006, 15)


In [31]:
df.head()

,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao,data,hora,minuto,dia_semana,latencia_envio_s,latencia_servidor_s
0,D33258,-22.91118,-43.19123,2026-04-07 15:34:04-03:00,19,383,2026-04-07 15:34:07-03:00,2026-04-07 15:34:08-03:00,2026-04-07 16:34:17,2026-04-07,15,34,Tuesday,3.0,1.0
1,D33175,-22.87352,-43.46544,2026-04-07 15:34:03-03:00,16,812,2026-04-07 15:34:07-03:00,2026-04-07 15:34:08-03:00,2026-04-07 16:34:17,2026-04-07,15,34,Tuesday,4.0,1.0
2,D33149,-22.89648,-43.61057,2026-04-07 15:34:03-03:00,31,757,2026-04-07 15:34:07-03:00,2026-04-07 15:34:08-03:00,2026-04-07 16:34:17,2026-04-07,15,34,Tuesday,4.0,1.0
3,D33164,-22.83070,-43.34547,2026-04-07 15:34:03-03:00,17,753,2026-04-07 15:34:07-03:00,2026-04-07 15:34:08-03:00,2026-04-07 16:34:17,2026-04-07,15,34,Tuesday,4.0,1.0
4,B58057,-22.90564,-43.20857,2026-04-07 15:33:58-03:00,36,312,2026-04-07 15:34:07-03:00,2026-04-07 15:34:11-03:00,2026-04-07 16:34:17,2026-04-07,15,33,Tuesday,9.0,4.0


Com base no fato de que temos dados que não sempre atualizados em tempo real, termos organizado a questão do timestamp, termos latitude e longitude e conseguirmos criar novas colunas com dados que podem nos ajudar em análises mais complexas, percebemos alguns insights:



1.   Podemos monitorar pontos de trânsito dentro da cidade, inclusive conseguindo apontar o ponto específico por conta da latitude e longitude.
2.   Podemos também monitorar linhas de transporte público, identificando mudanças que poderiam acontecer, como remoção ou adição de novas linhas.
3. Podemos criar gráficos para identificar horários de pico e velocidade média de cada local, gerando insights de melhoria de infraestrutura da cidade e mobilidade urbana.

